# Construct sentences from transcription

In [1]:
import json
from glob import glob

all_sentences_in_vid = {}
trans_paths = glob(r'..\additional_data\transcript_vid\*.json')
for path in trans_paths:
    code = path.split('\\')[-1].split('.')[0]
    words = []
    with open(path, 'r', encoding='utf-8') as f:
        words = json.load(f)

    sentences, selected = [], []
    s, e = 0.0, 0.0
    for word in words:
        e = word['end_time']
        selected.append(word['text'])
        if word['text'][-1] in [',', '.']:
            sentence = ' '.join(selected)
            # print(f'{s:.2f}', f'{e:.2f}',sentence)
            sentences.append({'start_time': s, 'end_time': e, 'text': sentence})

            selected = []
            s = e
    if selected:
        sentence = ' '.join(selected)
        sentences.append({'start_time': s, 'end_time': e, 'text': sentence})

    all_sentences_in_vid[code] = sentences

# Construct transcription phrases for keyframes

In [2]:
import csv

TIME_DEVIATION = 5
all_sentences_in_img = {}
csv_paths = glob(r'D:\AIC_LITE\mapkfs\*.csv')
for path in csv_paths:
    code = path.split('\\')[-1].split('.')[0]
    buffer = {}
    with open(path, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader)


        for row in reader:
            frame_id, time_order, fps, _ = row
            frame_id = int(frame_id)
            time_order = float(time_order)
            fps = float(fps)
            lower_bound = time_order - TIME_DEVIATION
            upper_bound = time_order + TIME_DEVIATION

            selected = [
                sentence['text'] for sentence in all_sentences_in_vid.get(code, [])
                if sentence['start_time'] >= lower_bound
                and sentence['end_time'] <= upper_bound
            ]

            test_sample = [
                sentence for sentence in all_sentences_in_vid.get(code, [])
                if sentence['start_time'] >= lower_bound
                and sentence['end_time'] <= upper_bound
            ]

            buffer[f'{code}_{frame_id:03d}'] = ' '.join(selected)
            # print(selected)
    all_sentences_in_img.update(buffer)

In [3]:
for code, transcript in list(all_sentences_in_img.items()):
    with open(f'../additional_data/transcript_img/{code}.txt', 'w', encoding='utf-8') as f:
        f.write(transcript)

# Construct detection phrases for images

In [4]:
obj_paths = glob('..\\additional_data\\obj_det\\*.json')
det_strs = {}
for path in obj_paths:
    code = path.split('\\')[-1].split('.')[0]
    dets = []
    with open(path, 'r') as f:
        dets = json.load(f)

    cnt_buffer = {}
    for det in dets:
        label = det['label']
        if label not in cnt_buffer:
            cnt_buffer[label] = 1
        else:
            cnt_buffer[label] += 1
        
    det_str = ', '.join([f'{cnt} {label}{'s' if cnt > 1 else ''}' for label, cnt in list(cnt_buffer.items())])
    # print(det_str)
    det_strs[code] = det_str

In [6]:
det_strs['L01_V011_071']

'12 persons, 2 cups, 4 bottles, 6 chairs'

# Model initialization

In [5]:
import torch
from FlagEmbedding import BGEM3FlagModel

ckpt = "BAAI/bge-m3"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = BGEM3FlagModel(
    ckpt,
    normalize_embeddings=True,
    use_bf16=True,
    devices=device,
    return_dense=True,
    return_sparse=True,
    model_kwargs={"attn_implementation": "flash_attention_2"}
)

C:\Users\Bui Thien Nghia\AppData\Roaming\Python\Python314\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

# Construct sparse phrase for images & encode

In [6]:
import numpy as np
from tqdm import tqdm

img_codes = list(all_sentences_in_img.keys())
sparse_text = [f'TRANSCRIPT: {all_sentences_in_img[code]} OBJECTS: {det_strs[code]}' for code in img_codes]

BATCH_SIZE = 256
feats = model.encode(
    sparse_text,
    batch_size=BATCH_SIZE,
)

Inference Embeddings: 100%|██████████| 32/32 [00:09<00:00,  3.22it/s]


In [7]:
for code, dense, sparse in tqdm(list(zip(img_codes, feats['dense_vecs'], feats['lexical_weights']))):
    sparse_token_id = np.array(list(sparse.keys()), dtype=np.int32)
    sparse_weights = np.array(list(sparse.values()), dtype=np.float32)
    np.savez(
        f'../additional_data/feat_bgem3/{code}.npz',
        dense=dense,
        sparse_token_ids=sparse_token_id,
        sparse_weights=sparse_weights,
    )

100%|██████████| 8115/8115 [00:04<00:00, 1869.44it/s]
